<img src="/Users/rajagopalvajja/PycharmProjects/OpenAIDemo/autogen_tutorial/conditional_1.jpg" width="300" height="200">

In [1]:
import autogen
import os
from dotenv import load_dotenv

_ = load_dotenv()


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
config_list = [
    {
        "model": "gpt-4o-mini",
        "api_key": os.environ.get("OPENAI_API_KEY", "your-api-key-here"),
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0,
}

In [5]:
# ============================================================
# STEP 1: Define the Agents
# ============================================================

# Router Agent - Analyzes and categorizes the query
router_agent = autogen.AssistantAgent(
    name="Router",
    system_message="""You are a routing agent. Analyze the user's message and
    categorize it into exactly ONE of these categories:
    - BILLING: payment issues, invoices, charges, subscriptions, refunds
    - TECHNICAL: login issues, bugs, errors, crashes, performance
    - GENERAL: anything else

    Respond with ONLY the category name (BILLING, TECHNICAL, or GENERAL)
    followed by a brief reason.""",
    llm_config=llm_config,
)

In [6]:
# Billing Agent
billing_agent = autogen.AssistantAgent(
    name="BillingAgent",
    system_message="""You are a billing support specialist. Help users with
    payment issues, invoices, charges, subscriptions, and refunds.
    Be helpful and professional. End your response with 'TERMINATE' when done.""",
    llm_config=llm_config,
)


In [7]:
# Technical Support Agent
tech_agent = autogen.AssistantAgent(
    name="TechAgent",
    system_message="""You are a technical support specialist. Help users with
    login issues, bugs, errors, crashes, and performance problems.
    Provide step-by-step troubleshooting. End your response with 'TERMINATE' when done.""",
    llm_config=llm_config,
)

In [8]:
# General Agent
general_agent = autogen.AssistantAgent(
    name="GeneralAgent",
    system_message="""You are a general customer support agent. Help users with
    general inquiries. Be friendly and helpful.
    End your response with 'TERMINATE' when done.""",
    llm_config=llm_config,
)


In [9]:
# User Proxy (represents the human)
user_proxy = autogen.UserProxyAgent(
    name="Customer",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    is_termination_msg=lambda x: "TERMINATE" in (x.get("content", "") or ""),
)

In [10]:
# ============================================================
# STEP 2: Define the Conditional Edge (Custom Speaker Selection)
# ============================================================

def custom_speaker_selection(last_speaker, groupchat):
    """
    This function implements CONDITIONAL EDGES.
    It decides which agent speaks next based on conversation content.
    """
    messages = groupchat.messages

    # Step A: Customer just spoke → Route to Router first
    if last_speaker.name == "Customer":
        print("🔀 [EDGE] Customer → Router (analyzing query...)")
        return router_agent

    # Step B: Router just categorized → Route to appropriate specialist
    if last_speaker.name == "Router":
        last_message = messages[-1]["content"].upper()

        if "BILLING" in last_message:
            print("🔀 [CONDITIONAL EDGE] Router → BillingAgent")
            return billing_agent
        elif "TECHNICAL" in last_message:
            print("🔀 [CONDITIONAL EDGE] Router → TechAgent")
            return tech_agent
        else:
            print("🔀 [CONDITIONAL EDGE] Router → GeneralAgent")
            return general_agent

    # Step C: Specialist responded → End conversation
    if last_speaker.name in ["BillingAgent", "TechAgent", "GeneralAgent"]:
        print("🔀 [EDGE] Specialist → End")
        return None  # End the conversation

    return None

In [11]:
# ============================================================
# STEP 3: Create GroupChat with Conditional Routing
# ============================================================

groupchat = autogen.GroupChat(
    agents=[user_proxy, router_agent, billing_agent, tech_agent, general_agent],
    messages=[],
    max_round=5,
    speaker_selection_method=custom_speaker_selection,  # ← CONDITIONAL EDGE HERE
)


In [12]:
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)



In [13]:
user_proxy.initiate_chat(
    manager,
    message="I can't login to my account. It keeps showing error 403.",
)



Customer (to chat_manager):

I can't login to my account. It keeps showing error 403.

--------------------------------------------------------------------------------
🔀 [EDGE] Customer → Router (analyzing query...)

Next speaker: Router

Router (to chat_manager):

TECHNICAL: The user is experiencing a login issue with an error message.

--------------------------------------------------------------------------------
🔀 [CONDITIONAL EDGE] Router → TechAgent

Next speaker: TechAgent

TechAgent (to chat_manager):

Error 403 typically indicates that access to the requested resource is forbidden. This can happen for several reasons. Let's go through some troubleshooting steps to resolve your login issue:

### Step 1: Check Your Credentials
1. **Verify Username and Password**: Ensure that you are entering the correct username and password. Check for any typos or case sensitivity.

### Step 2: Clear Browser Cache and Cookies
1. **Open your browser settings**.
2. **Find the option for clearing

ChatResult(chat_id=134982571097370156873459831708336585804, chat_history=[{'content': "I can't login to my account. It keeps showing error 403.", 'role': 'assistant', 'name': 'Customer'}, {'content': 'TECHNICAL: The user is experiencing a login issue with an error message.', 'name': 'Router', 'role': 'user'}, {'content': "Error 403 typically indicates that access to the requested resource is forbidden. This can happen for several reasons. Let's go through some troubleshooting steps to resolve your login issue:\n\n### Step 1: Check Your Credentials\n1. **Verify Username and Password**: Ensure that you are entering the correct username and password. Check for any typos or case sensitivity.\n\n### Step 2: Clear Browser Cache and Cookies\n1. **Open your browser settings**.\n2. **Find the option for clearing browsing data** (this may be under Privacy or Security settings).\n3. **Select Cookies and Cached Images** and clear them.\n4. **Restart your browser** and try logging in again.\n\n### 

In [14]:
# Reset for new conversation
groupchat.messages.clear()
user_proxy.initiate_chat(
    manager,
    message="I was charged twice for my subscription last month. I need a refund.",
)



Customer (to chat_manager):

I was charged twice for my subscription last month. I need a refund.

--------------------------------------------------------------------------------
🔀 [EDGE] Customer → Router (analyzing query...)

Next speaker: Router

Router (to chat_manager):

BILLING: The user is reporting a double charge and requesting a refund, which pertains to payment issues.

--------------------------------------------------------------------------------
🔀 [CONDITIONAL EDGE] Router → BillingAgent

Next speaker: BillingAgent

BillingAgent (to chat_manager):

I apologize for the inconvenience you've experienced with being charged twice for your subscription. To assist you further, could you please provide me with the following details?

1. The date of the charges.
2. The amount charged.
3. The payment method used.

Once I have this information, I can help you process the refund. Thank you for your patience! 

TERMINATE

-------------------------------------------------------------

ChatResult(chat_id=332391401001471527914334751845810578449, chat_history=[{'content': 'I was charged twice for my subscription last month. I need a refund.', 'role': 'assistant', 'name': 'Customer'}, {'content': 'BILLING: The user is reporting a double charge and requesting a refund, which pertains to payment issues.', 'name': 'Router', 'role': 'user'}, {'content': "I apologize for the inconvenience you've experienced with being charged twice for your subscription. To assist you further, could you please provide me with the following details?\n\n1. The date of the charges.\n2. The amount charged.\n3. The payment method used.\n\nOnce I have this information, I can help you process the refund. Thank you for your patience! \n\nTERMINATE", 'name': 'BillingAgent', 'role': 'user'}], summary="I apologize for the inconvenience you've experienced with being charged twice for your subscription. To assist you further, could you please provide me with the following details?\n\n1. The date of the c

In [15]:
# Reset for new conversation
groupchat.messages.clear()

user_proxy.initiate_chat(
    manager,
    message="What are your business hours?",
)

Customer (to chat_manager):

What are your business hours?

--------------------------------------------------------------------------------
🔀 [EDGE] Customer → Router (analyzing query...)

Next speaker: Router

Router (to chat_manager):

GENERAL: The inquiry is about business hours, which does not fall under billing or technical issues.

--------------------------------------------------------------------------------
🔀 [CONDITIONAL EDGE] Router → BillingAgent

Next speaker: BillingAgent

BillingAgent (to chat_manager):

Thank you for your inquiry! However, I specialize in assisting with billing issues, invoices, charges, subscriptions, and refunds. If you have any questions related to those topics, feel free to ask! 

TERMINATE

--------------------------------------------------------------------------------
🔀 [EDGE] Specialist → End

>>>>>>>> TERMINATING RUN (21dd9321-6408-4272-ad27-fc7f0eb7a0a5): No next speaker selected


ChatResult(chat_id=208832919153395086645252684606484063312, chat_history=[{'content': 'What are your business hours?', 'role': 'assistant', 'name': 'Customer'}, {'content': 'GENERAL: The inquiry is about business hours, which does not fall under billing or technical issues.', 'name': 'Router', 'role': 'user'}, {'content': 'Thank you for your inquiry! However, I specialize in assisting with billing issues, invoices, charges, subscriptions, and refunds. If you have any questions related to those topics, feel free to ask! \n\nTERMINATE', 'name': 'BillingAgent', 'role': 'user'}], summary='Thank you for your inquiry! However, I specialize in assisting with billing issues, invoices, charges, subscriptions, and refunds. If you have any questions related to those topics, feel free to ask! \n\n', cost={'usage_including_cached_inference': {'total_cost': 0}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_input=[])